# ICM vs Chip EV

Compares ICM equity to chip-equity-based decisions across different tournament
situations. Demonstrates how bubble pressure distorts ranges and how chip chop
hybrid models interpolate between pure ICM and pure chip equity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from speculator_ev_engine.poker.icm import (
    malmuth_harville_icm, chip_chop_icm, bubble_factor, icm_push_fold_ev
)

## 1. ICM vs Chip Chop: 3-player bubble

In [ ]:
# 3 players remaining, typical SNG payout
stacks = np.array([5000.0, 3000.0, 2000.0])
payouts = np.array([500.0, 300.0, 200.0])

icm_result = malmuth_harville_icm(stacks, payouts)
chip_chop_result = chip_chop_icm(stacks, payouts, blend_weight=0.0)  # pure chip chop
hybrid_result = chip_chop_icm(stacks, payouts, blend_weight=0.5)     # 50/50 blend

labels = ['Big Stack', 'Medium Stack', 'Short Stack']
x = np.arange(len(stacks))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width, icm_result.equities, width, label='ICM')
ax.bar(x, hybrid_result.equities, width, label='50/50 Hybrid')
ax.bar(x + width, chip_chop_result.equities, width, label='Chip Chop')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Dollar Equity')
ax.set_title('ICM vs Hybrid vs Chip Chop: 3-Player SNG Bubble')
ax.legend()
plt.show()

print('ICM equities:', icm_result.equities)
print('Sum check:', np.sum(icm_result.equities), '==', icm_result.total_prize_pool)

## 2. Bubble Factor Heat Map

In [ ]:
# How bubble factor changes with stack depth
total_chips = 10000.0
stack_ratios = np.linspace(0.1, 0.8, 20)
bubble_factors = []

for sr in stack_ratios:
    hero = sr * total_chips
    opp1 = (1.0 - sr) * 0.6 * total_chips
    opp2 = (1.0 - sr) * 0.4 * total_chips
    stk = np.array([hero, opp1, opp2])
    bf = bubble_factor(stk, payouts, player_index=0)
    bubble_factors.append(bf.bubble_factor)

plt.figure(figsize=(10, 6))
plt.plot(stack_ratios * 100, bubble_factors, 'b-o')
plt.axhline(1.0, color='red', linestyle='--', label='Chip EV threshold')
plt.xlabel('Hero Stack (% of total)')
plt.ylabel('Bubble Factor')
plt.title('Bubble Factor vs Stack Depth (3-player SNG)')
plt.legend()
plt.show()

## 3. Chip Chop Blend Sweep

In [ ]:
# Short stack equity under different blend weights
blend_weights = np.linspace(0.0, 1.0, 20)
short_stack_equities = []

for bw in blend_weights:
    result = chip_chop_icm(stacks, payouts, blend_weight=bw)
    short_stack_equities.append(result.equities[2])  # short stack

plt.figure(figsize=(10, 6))
plt.plot(blend_weights, short_stack_equities, 'g-o')
plt.xlabel('Blend Weight (0=ChipChop, 1=ICM)')
plt.ylabel('Short Stack Equity ($)')
plt.title('Short Stack Equity vs ICM/Chip Chop Blend')
plt.show()

## 4. Push/Fold EV at the Bubble

In [ ]:
# Hero is short stack on the button, 5BB
stk = np.array([2000.0, 5000.0, 3000.0])
result = icm_push_fold_ev(
    stacks=stk,
    payouts=payouts,
    hero_index=0,
    hero_hand_strength=0.55,
    caller_indices=[1, 2],
    caller_call_frequencies=np.array([0.15, 0.25]),
    caller_hand_ranges=np.array([0.45, 0.40]),
    big_blind=100.0,
)
print(f'Push EV: ${result["ev_push"]:.2f}')
print(f'Fold EV: ${result["ev_fold"]:.2f}')
print(f'Push is correct: {result["ev_diff"] > 0}')
print(f'EV diff: ${result["ev_diff"]:.2f}')